In [3]:
import requests
import pandas as pd
import datetime

# Test NASA POWER for Kandy

In [14]:
import requests
import pandas as pd
import datetime

lat = 7.2906
lon = 80.6337

start_date = "20200101"
# Temporarily set end_date to a known past date for debugging purposes
end_date = "20260831"

url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point" # Corrected endpoint
    "?parameters=T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN,WS10M"
    f"&community=AG" # Re-added community parameter
    f"&longitude={lon}"
    f"&latitude={lat}"
    f"&start={start_date}"
    f"&end={end_date}"
    "&format=JSON"
)

response = requests.get(url)

print("Status:", response.status_code)

data = response.json()

print(data.keys())

Status: 200
dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


# Convert the result into a DataFrame

In [15]:
weather = data["properties"]["parameter"]

df = pd.DataFrame(weather)

df.head()

,T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN,WS10M
20200101,24.65,1.05,88.58,17.30,2.57
20200102,24.65,5.74,91.58,16.02,3.32
20200103,24.99,1.15,89.01,20.51,2.83
20200104,24.29,1.07,89.55,21.63,2.92
20200105,23.95,0.19,90.29,22.53,3.69


# Fix the date column

In [16]:
df = df.reset_index()

df = df.rename(columns={
    "index": "date",
    "T2M": "temperature_c",
    "PRECTOTCORR": "rainfall_mm",
    "RH2M": "humidity_percent",
    "ALLSKY_SFC_SW_DWN": "solar_radiation",
    "WS10M": "wind_speed_ms"
})

df["date"] = pd.to_datetime(df["date"])

df.head()

,date,temperature_c,rainfall_mm,humidity_percent,solar_radiation,wind_speed_ms
0,2020-01-01,24.65,1.05,88.58,17.30,2.57
1,2020-01-02,24.65,5.74,91.58,16.02,3.32
2,2020-01-03,24.99,1.15,89.01,20.51,2.83
3,2020-01-04,24.29,1.07,89.55,21.63,2.92
4,2020-01-05,23.95,0.19,90.29,22.53,3.69


# Add the district

In [19]:
import requests
import pandas as pd
import time

# ============================================================
# SRI LANKA - ALL 25 DISTRICTS
# Latitude and Longitude are representative points
# ============================================================

districts = {
    "Colombo": (6.9271, 79.8612),
    "Gampaha": (7.0840, 80.0098),
    "Kalutara": (6.5854, 79.9607),

    "Kandy": (7.2906, 80.6337),
    "Matale": (7.4675, 80.6234),
    "Nuwara Eliya": (6.9497, 80.7891),

    "Galle": (6.0329, 80.2168),
    "Matara": (5.9549, 80.5550),
    "Hambantota": (6.1429, 81.1212),

    "Jaffna": (9.6615, 80.0255),
    "Kilinochchi": (9.3803, 80.3770),
    "Mannar": (8.9810, 79.9044),
    "Mullaitivu": (9.2671, 80.8128),
    "Vavuniya": (8.7514, 80.4971),

    "Batticaloa": (7.7310, 81.6747),
    "Ampara": (7.2917, 81.6720),
    "Trincomalee": (8.5874, 81.2152),

    "Kurunegala": (7.4863, 80.3623),
    "Puttalam": (8.0362, 79.8283),

    "Anuradhapura": (8.3114, 80.4037),
    "Polonnaruwa": (7.9403, 81.0188),

    "Badulla": (6.9934, 81.0550),
    "Monaragala": (6.8728, 81.3507),

    "Ratnapura": (6.6828, 80.3992),
    "Kegalle": (7.2513, 80.3464)
}


# ============================================================
# DATE RANGE
# ============================================================

START_DATE = "20000101"
END_DATE   = "20251231"


# ============================================================
# NASA POWER PARAMETERS
# ============================================================

PARAMETERS = (
    "T2M,"
    "T2M_MAX,"
    "T2M_MIN,"
    "PRECTOTCORR,"
    "RH2M,"
    "ALLSKY_SFC_SW_DWN,"
    "WS10M"
)


# ============================================================
# STORE DATA
# ============================================================

all_data = []


# ============================================================
# DOWNLOAD DATA FOR EACH DISTRICT
# ============================================================

for district, (latitude, longitude) in districts.items():

    print("\n==========================================")
    print("Downloading:", district)
    print("Latitude:", latitude)
    print("Longitude:", longitude)
    print("==========================================")

    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point" # Corrected endpoint
        f"?parameters={PARAMETERS}"
        "&community=AG"
        f"&longitude={longitude}"
        f"&latitude={latitude}"
        f"&start={START_DATE}"
        f"&end={END_DATE}"
        "&format=JSON"
    )

    try:

        response = requests.get(
            url,
            timeout=120
        )

        print("HTTP Status:", response.status_code)

        if response.status_code != 200:
            print("FAILED:", district)
            print(response.text[:500])
            continue

        data = response.json()

        # ----------------------------------------------------
        # Extract weather parameters
        # ----------------------------------------------------

        weather = data["properties"]["parameter"]

        df = pd.DataFrame(weather)

        # ----------------------------------------------------
        # Convert date index into column
        # ----------------------------------------------------

        df = df.reset_index()

        df = df.rename(
            columns={
                "index": "date"
            }
        )

        # ----------------------------------------------------
        # Rename NASA variables
        # ----------------------------------------------------

        df = df.rename(
            columns={
                "T2M": "temperature_avg_c",
                "T2M_MAX": "temperature_max_c",
                "T2M_MIN": "temperature_min_c",
                "PRECTOTCORR": "rainfall_mm",
                "RH2M": "humidity_percent",
                "ALLSKY_SFC_SW_DWN": "solar_radiation_mj_m2_day",
                "WS10M": "wind_speed_m_s"
            }
        )

        # ----------------------------------------------------
        # Convert date
        # ----------------------------------------------------

        df["date"] = pd.to_datetime(
            df["date"]
        )

        # ----------------------------------------------------
        # Add district information
        # ----------------------------------------------------

        df["district"] = district

        df["latitude"] = latitude

        df["longitude"] = longitude

        # ----------------------------------------------------
        # Add to main list
        # ----------------------------------------------------

        all_data.append(df)

        print(
            "SUCCESS:",
            district,
            "| Records:",
            len(df)
        )

        # Small delay to avoid sending requests too quickly
        time.sleep(1)


    except Exception as e:

        print(
            "ERROR:",
            district,
            "|",
            str(e)
        )


# ============================================================
# COMBINE ALL DISTRICTS
# ============================================================

print("\n\nCombining all districts...")

weather_df = pd.concat(
    all_data,
    ignore_index=True
)


# ============================================================
# REORDER COLUMNS
# ============================================================

weather_df = weather_df[
    [
        "date",
        "district",
        "latitude",
        "longitude",
        "temperature_avg_c",
        "temperature_max_c",
        "temperature_min_c",
        "rainfall_mm",
        "humidity_percent",
        "solar_radiation_mj_m2_day",
        "wind_speed_m_s"
    ]
]


# ============================================================
# SORT DATA
# ============================================================

weather_df = weather_df.sort_values(
    by=[
        "district",
        "date"
    ]
).reset_index(drop=True)


# ============================================================
# DISPLAY INFORMATION
# ============================================================

print("\n==========================================")
print("DOWNLOAD COMPLETED")
print("==========================================")

print(
    "Number of rows:",
    len(weather_df)
)

print(
    "Number of columns:",
    len(weather_df.columns)
)

print(
    "Number of districts:",
    weather_df["district"].nunique()
)

print(
    "Date range:",
    weather_df["date"].min(),
    "to",
    weather_df["date"].max()
)

print("\nDistricts downloaded:")

print(
    weather_df["district"].unique()
)

print("\nFirst 20 records:")

display(
    weather_df.head(20)
)


Downloading: Colombo
Latitude: 6.9271
Longitude: 79.8612
HTTP Status: 200
SUCCESS: Colombo | Records: 9497

Downloading: Gampaha
Latitude: 7.084
Longitude: 80.0098
HTTP Status: 200
SUCCESS: Gampaha | Records: 9497

Downloading: Kalutara
Latitude: 6.5854
Longitude: 79.9607
HTTP Status: 200
SUCCESS: Kalutara | Records: 9497

Downloading: Kandy
Latitude: 7.2906
Longitude: 80.6337
HTTP Status: 200
SUCCESS: Kandy | Records: 9497

Downloading: Matale
Latitude: 7.4675
Longitude: 80.6234
HTTP Status: 200
SUCCESS: Matale | Records: 9497

Downloading: Nuwara Eliya
Latitude: 6.9497
Longitude: 80.7891
HTTP Status: 200
SUCCESS: Nuwara Eliya | Records: 9497

Downloading: Galle
Latitude: 6.0329
Longitude: 80.2168
HTTP Status: 200
SUCCESS: Galle | Records: 9497

Downloading: Matara
Latitude: 5.9549
Longitude: 80.555
HTTP Status: 200
SUCCESS: Matara | Records: 9497

Downloading: Hambantota
Latitude: 6.1429
Longitude: 81.1212
HTTP Status: 200
SUCCESS: Hambantota | Records: 9497

Downloading: Jaffna
Lat

,date,district,latitude,longitude,temperature_avg_c,temperature_max_c,temperature_min_c,rainfall_mm,humidity_percent,solar_radiation_mj_m2_day,wind_speed_m_s
0,2000-01-01,Ampara,7.2917,81.672,25.49,26.65,24.50,0.02,77.67,16.30,6.55
1,2000-01-02,Ampara,7.2917,81.672,25.61,26.47,24.96,0.04,88.04,10.51,7.15
2,2000-01-03,Ampara,7.2917,81.672,25.76,26.46,25.26,0.02,87.67,4.25,6.80
3,2000-01-04,Ampara,7.2917,81.672,25.45,26.16,25.06,0.01,88.50,12.88,6.98
4,2000-01-05,Ampara,7.2917,81.672,25.81,26.66,25.12,0.07,88.59,11.23,6.26
5,2000-01-06,Ampara,7.2917,81.672,25.91,26.88,25.21,7.25,88.92,11.05,5.41
6,2000-01-07,Ampara,7.2917,81.672,25.87,26.91,25.29,5.25,87.99,11.40,6.79
7,2000-01-08,Ampara,7.2917,81.672,25.85,26.54,25.19,0.72,88.26,14.27,5.46
8,2000-01-09,Ampara,7.2917,81.672,25.69,27.01,24.72,4.07,87.68,15.76,4.10
9,2000-01-10,Ampara,7.2917,81.672,25.97,26.97,25.04,7.09,87.55,17.65,4.62


# Check whether all 25 districts were downloaded

In [20]:
print("Number of districts:", weather_df["district"].nunique())

print("\nDistrict list:")
for district in sorted(weather_df["district"].unique()):
    print(district)

Number of districts: 25

District list:
Ampara
Anuradhapura
Badulla
Batticaloa
Colombo
Galle
Gampaha
Hambantota
Jaffna
Kalutara
Kandy
Kegalle
Kilinochchi
Kurunegala
Mannar
Matale
Matara
Monaragala
Mullaitivu
Nuwara Eliya
Polonnaruwa
Puttalam
Ratnapura
Trincomalee
Vavuniya


# Check the amount of data

In [21]:
weather_df.groupby("district").size()

,0
district,
Ampara,9497
Anuradhapura,9497
Badulla,9497
Batticaloa,9497
Colombo,9497
Galle,9497
Gampaha,9497
Hambantota,9497
Jaffna,9497


In [22]:
# Check missing values
print("Missing values:")
print(weather_df.isnull().sum())

Missing values:
date                         0
district                     0
latitude                     0
longitude                    0
temperature_avg_c            0
temperature_max_c            0
temperature_min_c            0
rainfall_mm                  0
humidity_percent             0
solar_radiation_mj_m2_day    0
wind_speed_m_s               0
dtype: int64


In [23]:
missing_percentage = (
    weather_df.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_percentage)

date                         0.0
district                     0.0
latitude                     0.0
longitude                    0.0
temperature_avg_c            0.0
temperature_max_c            0.0
temperature_min_c            0.0
rainfall_mm                  0.0
humidity_percent             0.0
solar_radiation_mj_m2_day    0.0
wind_speed_m_s               0.0
dtype: float64


# Check the weather values

In [24]:
weather_df.describe()

,date,latitude,longitude,temperature_avg_c,temperature_max_c,temperature_min_c,rainfall_mm,humidity_percent,solar_radiation_mj_m2_day,wind_speed_m_s
count,237425,237425.000000,237425.00000,237425.000000,237425.000000,237425.000000,237425.000000,237425.000000,237425.000000,237425.000000
mean,2012-12-31 00:00:00.000000256,7.586436,80.58856,26.558000,29.585956,24.319077,4.752410,81.090404,19.198766,4.636669
min,2000-01-01 00:00:00,5.954900,79.82830,17.270000,21.270000,10.600000,0.000000,48.940000,1.040000,0.320000
25%,2006-07-02 00:00:00,6.927100,80.21680,25.400000,28.040000,22.620000,0.190000,76.600000,16.990000,3.040000
50%,2012-12-31 00:00:00,7.291700,80.49710,26.680000,29.220000,24.840000,1.480000,81.880000,20.050000,4.470000
75%,2019-07-02 00:00:00,8.311400,81.01880,27.930000,30.940000,26.280000,5.800000,86.230000,22.340000,6.050000
max,2025-12-31 00:00:00,9.661500,81.67470,32.750000,41.500000,30.550000,294.220000,97.560000,27.630000,15.040000
std,NaN,1.034769,0.53295,1.934465,2.219842,2.520346,8.639944,6.802399,4.283449,2.018254


# Save as CSV

In [25]:
file_name = "Sri_Lanka_NASA_POWER_Weather_2000_2025.csv"

weather_df.to_csv(
    file_name,
    index=False
)

print("CSV created:")
print(file_name)

CSV created:
Sri_Lanka_NASA_POWER_Weather_2000_2025.csv


In [26]:
from google.colab import files

files.download(
    "Sri_Lanka_NASA_POWER_Weather_2000_2025.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>